# MCP Fundamentals

*Notebook 25*

Connect your agent to MCP servers and get their tools automatically: no per-tool integration code.


---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

# Notebook-specific imports
import subprocess
import shutil

MODEL = "gpt-5-mini"

print("✅ Imports ready!")

Agent runs are traced by default.

With sensitive-data capture enabled, the dashboard shows inputs, outputs, and MCP tool activity together.

### Check Node.js / npx

The filesystem MCP server runs via `npx`, so verify Node.js is installed.

Note: this is specific to the demo server. Other MCP servers use other runtimes.

In [ ]:
# Check Node.js and npx availability
try:
    node_version = subprocess.check_output(["node", "--version"], text=True).strip()
    npx_version = subprocess.check_output(["npx", "--version"], text=True).strip()
    print(f"✅ Node.js: {node_version}")
    print(f"✅ npx: {npx_version}")
    print("\n✅ Ready for MCP!")
except FileNotFoundError:
    print("❌ Node.js not found")
    print("\nInstall Node.js from https://nodejs.org (LTS version)")
    print("After installing, restart your kernel and re-run this cell.")

---

## 🔌 Part 1: What Is MCP?

**MCP (Model Context Protocol)** is an open standard for connecting AI agents to external tools and data sources.

Think of it like USB-C for AI:
- Before USB-C, every device had its own connector: constant adapter chaos
- USB-C is one shared connector for compatible devices
- Before MCP, AI tools often needed custom integration code
- MCP is one shared protocol for compatible agents and tools

**Three ways to give an agent capabilities:**

- `@function_tool`: you write the code, the tool runs locally in Python

- Built-in tools (e.g. web search): OpenAI hosts and runs them, zero code needed

- MCP: a separate MCP server hosts the tool (running locally or remotely), the SDK connects and exposes its tools to the agent

MCP is the third path: no per-tool integration code needed.

**The key insight:** an MCP server exposes tools in a standard format. A compatible client can discover a server's tools and use them without knowing anything about how the server works internally.

### The Three Pieces

```
MCP Server   : exposes tools (filesystem, web, database, etc.)
MCP Client   : connects to the server (the Agents SDK is the client)
Agent        : uses the tools the server exposes
```

You write no per-tool wrapper code.

The server handles the implementation.

The agent handles the reasoning.

MCP handles the connection.

MCP servers can also expose **resources** (read-only data) and reusable prompts.

Only tools reach the agent automatically via `mcp_servers`: resources and prompts require explicit application code (`list_resources()` / `read_resource()`, `list_prompts()` / `get_prompt()`).

---

## 📁 Part 2: Your First MCP Server: Filesystem

The filesystem MCP server gives the agent tools to read, write, and navigate files.

It runs as a local subprocess via `npx`.

We'll use `MCPServerStdio`, the SDK's client for local servers that communicate over standard input/output (stdio).

Calling `list_tools()` inspects a new MCP server's surface before trusting an agent with it.

In [ ]:
# Create a clean sample workspace for the agent (remove any stale copy first)
workspace = Path("mcp_workspace").resolve()
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir()

(workspace / "notes.txt").write_text("""Project Notes
==============
- MCP connects agents to external tools
- The filesystem server exposes read/write tools
- No per-tool wrapper code needed
- Compatible agents can discover these tools automatically
""")

(workspace / "tasks.txt").write_text("""TODO List
=========
1. Learn MCP fundamentals
2. Connect to filesystem server
3. Try web fetch server
4. Build MCP-powered assistant
""")

(workspace / "config.json").write_text('{"version": "1.0", "debug": false, "max_retries": 3}')

# --------------------------------------------------------------
print("✅ Workspace created")
print(f"    Location: {workspace.parent.name}/{workspace.name}")
print(f"    Files: {[f.name for f in workspace.iterdir()]}")

#### Connect and Discover Tools

In [ ]:
print("🔌 CONNECTING TO FILESYSTEM MCP SERVER")
print("=" * 60)

async with MCPServerStdio(
    name="Filesystem",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem@2026.7.10", str(workspace.resolve())]
    },
    cache_tools_list=True,
    client_session_timeout_seconds=30  # first run may download the pinned package
) as fs_server:

    # List available tools: the server tells us what it can do
    tools = await fs_server.list_tools()
    print(f"✅ Connected! {len(tools)} tools available:\n")
    for tool in tools:
        print(f"  🔧 {tool.name}")
        if hasattr(tool, 'description') and tool.description:
            print(f"     {tool.description[:80]}")

print("=" * 60)

⚠️ **Security note:** Scope the filesystem MCP server to a dedicated workspace: never point it at your home directory or project root.

This notebook configures no approval or tool filter, so every discovered tool, including writes, can execute without a pause. The dedicated synthetic workspace is the safety boundary here.

`MCPServerStdio` supports `require_approval` and `tool_filter` for gating: Notebook 27 uses `require_approval` to protect destructive operations.

💡 **Tip:** Pass `cache_tools_list=True` to `MCPServerStdio` to skip re-fetching the tool list on every run.

Cached definitions can go stale: call `invalidate_tools_cache()` if the server's tools change.

---

## 🤖 Part 3: Agent with MCP Tools

Pass the MCP server to an agent via `mcp_servers=[server]`.

The agent automatically sees all the tools the server exposes, including write tools.

The model decides which of them to call.

In [ ]:
print("🤖 AGENT WITH MCP FILESYSTEM TOOLS")
print("=" * 60)

async with MCPServerStdio(
    name="Filesystem",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem@2026.7.10", str(workspace.resolve())]
    },
    cache_tools_list=True,
    client_session_timeout_seconds=30  # first run may download the pinned package
) as fs_server:

    agent = Agent(
        name="FileAssistant",
        instructions=(
            "You are a file assistant with access to a workspace directory.\n"
            "Use the filesystem tools to read, list, and summarize files when asked."
        ),
        model=MODEL,
        mcp_servers=[fs_server]
    )

    # Ask the agent to explore the workspace
    result = await Runner.run(agent, input="List all files in the workspace and give me a brief summary of what each one contains.")
    print(result.final_output)
    called = [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"]
    print(f"🔧 MCP tools this run called: {called}")
    if not called:
        print("⚠️  No MCP tools were called in this run")

    # ---- Now let's write a file ----
    print("\n✍️ WRITE DEMO")
    print("-" * 60)

    summary_path = workspace / "summary.txt"
    print(f"summary.txt exists before the run: {summary_path.exists()}")

    result = await Runner.run(agent, input="Read tasks.txt and create a new file called summary.txt with a one-sentence summary of the task list.")
    print(result.final_output)
    called = [item.raw_item.name for item in result.new_items if item.type == "tool_call_item"]
    print(f"🔧 MCP tools this run called: {called}")
    if "write_file" not in called:
        print("⚠️  write_file was not called in this run")

# Verify outside the MCP context: the file persists on disk
if summary_path.exists():
    print(f"\n✅ summary.txt created by this run:")
    print(summary_path.read_text())
else:
    print("\n⚠️  summary.txt was not created")

print("=" * 60)

To use a different local stdio server, change the `command` and `args`: they come from each server's documentation.

For a remote server using Streamable HTTP, use `MCPServerStreamableHttp` instead of `MCPServerStdio`.

After connection, the `mcp_servers=[server]` pattern is the same either way.

---

#### Cleanup

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)
print("✅ Workspace removed" if not workspace.exists() else "⚠️ Workspace still present")

---

## 💪 Practice Exercises

### Exercise 1: File Organizer

*Covers: MCP filesystem server: file operations via MCP*

Create a workspace with several files and ask the agent to organize or summarize them.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: File Organizer
# --------------------------------------------------------------
# Objective: Use the filesystem MCP server to analyze a set of files.

# TODO 1: Create a fresh, dedicated workspace directory (remove any
#           stale copy first) with at least 3 text files containing
#           different types of content (meeting notes, a shopping
#           list, a project plan, etc.)

# TODO 2: Connect to MCPServerStdio with the filesystem server
#           pointing at your workspace

# TODO 3: Create an agent with the MCP server

# TODO 4: Confirm index.txt does not exist yet, then ask the agent
#           to read all files and create a consolidated index.txt
#           listing each file with a one-sentence description

# TODO 5: Print the MCP tools the run called (result.new_items,
#           item.type == "tool_call_item") and check that at least
#           one MCP tool ran and that write_file is among them,
#           then verify index.txt exists and print its contents

# TODO 6: Clean up the workspace and verify it is gone

# --- Write your code below this line ---

### Exercise 2: Discover Available Tools

*Covers: MCP introspection: listing available tools and descriptions*

Connect to the filesystem MCP server and print a formatted table of all available tools and their descriptions.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Tool Discovery
# --------------------------------------------------------------
# Objective: Explore what tools an MCP server exposes.

# TODO 1: Connect to MCPServerStdio pointed at a dedicated
#           disposable directory (never your home or project root)

# TODO 2: Call server.list_tools() and store the result

# TODO 3: Print each tool's name and full description
#           formatted clearly (e.g., "Tool: read_file\n  Description: ...")

# TODO 4: Print the total count of available tools

# TODO 5: Remove the disposable directory and verify it is gone

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**MCP is the USB-C for AI tools:**

- One standard protocol: a compatible client discovers a server's declared tools automatically

- No per-tool wrappers: the SDK converts discovered server tools into agent tools

- A growing ecosystem of pre-built MCP servers for common services
<br>
<br>

**`MCPServerStdio` runs a local server as a subprocess:**

- Remote Streamable HTTP servers use `MCPServerStreamableHttp` with the same agent pattern
<br>
<br>

**Manage the server with `async with` and pass it via `mcp_servers`:**

- Always use `async with MCPServerStdio(...) as server:` (manages the connection lifecycle)

- Pass `mcp_servers=[server]` to the agent: tools appear automatically

- `cache_tools_list=True` skips repeated discovery, `invalidate_tools_cache()` refreshes it

---

## 📍 Next Step

**Notebook 26: Real-World MCP Servers**  

Connect to the filesystem and web fetch MCP servers and combine multiple MCP servers in one agent.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-25-mcp-fundamentals)

---